In [1]:
!pip install -q peft==0.11.1
!pip install -q bitsandbytes==0.43.1
!pip install peft==0.11.1
!pip install fastchat
!pip install --upgrade transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 134.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2


In [2]:
#Load Model2
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
from transformers import AutoConfig, AutoTokenizer

model_dir = "/content/drive/MyDrive/Scalable ML/Lab2/model_1_a"
cfg = AutoConfig.from_pretrained(model_dir)
print("model_type:", getattr(cfg, "model_type", None))

tok = AutoTokenizer.from_pretrained(model_dir)
print("Tokenizer:", type(tok))

from transformers import AutoModelForCausalLM
import torch

# Ladda modellen för inferens
model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.float16)
model.eval()  # Viktigt: inferensläge
model.to("cuda")

prompt = "Write a short poem"

# Tokenisera input med max sekvenslängd
inputs = tok(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2048
).to("cuda")

# LLaMA behöver inte token_type_ids
if "token_type_ids" in inputs:
    del inputs["token_type_ids"]

# -----------------------------
# Generera svar
# -----------------------------
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,      # Hur många nya tokens modellen ska generera
        do_sample=True,          # Sätter sampling
        temperature=0.1          # Sätter temperatur för mest deterministiska svar
    )

# -----------------------------
# Avkoda och skriv ut
# -----------------------------
response = tok.decode(output[0], skip_special_tokens=True)
print("Model response:", response)


model_type: llama
Tokenizer: <class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>
Model response: Write a short poem about the concept of love.
